In [1]:
# Kernel SVM. This was by far the hardest one for me.
# It is based off numerous mathematical theorems and properties.
# It starts off simple. We have two classes that are not linearly separable, so how can we separate them with SVM?
# Linear SVM will not achieve a good score, but it can still be used. How? We "mimic" the transition of the current data to a superior plane
# In this new plane, the data will be linearly separable! However, it is important to understand it's not a "straight line" 
# There are a few issues, the main one being the fact that some functions (named kernel functions in this case) can go to a very high dimension
# How to avoid this? Kernel trick!
# First, tho, let's understand how kernel svm works. It starts from linear svm main function : f(x) = wT @ x + b, f(x) = 0;
# Class A is f(x) = 1, class B is f(x) = -1. Now, we multiply to determine scores : yi(fx) >= 1 ideally.
# Now, find the distance : let x1,x2 s.t. f(x1) = 1, f(x2) = -1, w' = direction unitary vector for w
# Subtract, getting wT(x1-x2) = 2; but w' = w/norm(w) => distance = (x1-x2)w'; substitute and now we have the optimisation problem
# 2/norm(w) which is maximisation, equivalent to minimising 1/2 * norm(w)
# We also have the constraint yi*f(x) >= 1 <=> 1 - yi f(xi) >-= 0 noted as opt(xi, yi). 
# Mathematically, we can apply KKT conditions and Lagrange on the initial loss function.
# Now, we obtain the Lagrange func : L(w,bias,alpha) = 1/2 norm(w)^2 + sum alphai * opt(xi, yi)
# Derive for w, then beta and you find : DL/Dw = w - sum alphai*xi*yi, DL/Dbias = -sum alphai*yi
# For minimisation, from math theory, f'(x) = 0 => x is min/max for x in R. For R^n, this is a critical point and gradient and
# hessian must be evaluated. In our case, w = sum alphai xi yi and -sum alphai yi = 0.
# Now, alphai must be >= 0 so that if we make mistakes in lagrangian it can be very high and penalised
# Lets say we want to minimise f(x) arbitrarily chosen function, so minf(x) but we also have g(x) = 1-x <=0;
# Now we can make L(x, alpha) = f(x) + alpha * g(x), and we apply duality : maxmin L(x,alpha). If we have g(x) broken rule,
# L(x, alpha) can be +inf. so infinite 'punishment'
# Now apply maxmin on our function and we get a hard to write formula here, but in it appears xj * xiT
# So we take it to another plane, phi(xi) let's say. To avoid computing in R^n with n a very large value, we use kernel functions
# So on this we apply the kernel functions (because we want it linearly separable and because K(xi, xj) = our needed formula directly comptued)
# And boom! Done! We now just apply some KKT principles like primality (all rules must be followed)
# and complementarity (the product of each multiplier and its corresponding inequality must be 0)
import numpy as np
import kagglehub
import pandas as pd
x, y = None, None
dataset, model = None, None

def load_data():
    global x, y
    global dataset, model

    dataset = pd.read_csv("/kaggle/input/datasets/organizations/uciml/breast-cancer-wisconsin-data/data.csv")
    dataset = dataset.drop(columns=["id", "Unnamed: 32"])
    y = np.where(dataset["diagnosis"] == "M", 1.0, -1.0)
    dataset = dataset.drop(columns=["diagnosis"])
    x = np.array(dataset)

class Kernel_SVM():
    def __init__(self, gamma=0.1, C=1.0, eps=1e-3, max_passes=10):
        self.alphas = None
        self.gamma = gamma
        self.bias = 0.0
        self.x_tr, self.y_tr = None, None

        self.C = C
        self.eps = eps
        self.max_passes = max_passes

    def _euclidean_norm(self, vector):
        return np.sqrt(np.sum(abs(vector) ** 2))

    def _rbf_func(self, x1, x2):
        return np.exp(
            -(self.gamma * (self._euclidean_norm(x1 - x2)) ** 2)
        )

    def _kernel_matr(self, x):
        n = x.shape[0]
        K = np.zeros((n, n))

        for i in range(n):
            for j in range(n):
                K[i][j] = self._rbf_func(x[i], x[j])

        return K

    def fit(self, x, y):
        n = x.shape[0]
        self.alphas = np.zeros(n)

        self.x_tr, self.y_tr = x, y
        self.K = self._kernel_matr(x)

        # number of consecutive full passes with no alpha changes
        passes = 0

        while passes < self.max_passes:

            # number of alpha pairs changed during this pass
            num_changed_alphas = 0

            for i in range(n):
                # for each point up to the ith one, see the contribution of each point * the classes each point has * the influence between two points
                f_i = np.sum(self.alphas * self.y_tr * self.K[:, i]) + self.bias

                err_i = f_i - y[i]

                violates_kkt = (
                    # if we are in a bad spot and we can update alpha
                    (y[i] * err_i < -self.eps and self.alphas[i] < self.C)
                    or
                    # if we are in a good spot but it still contributes to the frontier through alpha
                    (y[i] * err_i > self.eps and self.alphas[i] > 0)
                )

                if not violates_kkt:
                    continue

                # since we are using SMO, it is enough to have only two points to update alpha_i
                j = i
                while j == i:
                    j = np.random.randint(0, n)

                f_j = np.sum(self.alphas * self.y_tr * self.K[:, j]) + self.bias

                err_j = f_j - y[j]

                old_i, old_j = self.alphas[i], self.alphas[j]

                # calculate the valid interval in which alpha_j can move
                # while keeping both alphas inside [0, C]
                if y[i] == y[j]:
                    lowest = max(0, old_i + old_j - self.C)
                    highest = min(self.C, old_i + old_j)
                else:
                    lowest = max(0, old_j - old_i)
                    highest = min(self.C, self.C + old_j - old_i)

                if lowest == highest:
                    continue

                # K[i,i] = K[j,j] = 1 for RBF
                eta = 2 * self.K[i][j] - 2

                if eta >= 0:
                    continue

                # modifies a_j by looking at how much the points differ (err_i - err_j) and the optimal direction which is eta
                self.alphas[j] = (old_j - y[j] * (err_i - err_j) / eta)
                self.alphas[j] = np.clip(self.alphas[j], lowest, highest)

                # ignore changes that are numerically insignificant
                if abs(self.alphas[j] - old_j) < self.eps:
                    self.alphas[j] = old_j
                    continue

                # yi a_i + yj a_j = ct = yi o_i + yj o_j
                # a_i = o_i + yj/yi (o_j - a_j), but y i and y j are +-1 so its correct
                self.alphas[i] = (old_i + y[i] * y[j] * (old_j - self.alphas[j]))

                # equivalent to yi - new_g(x) where g(x) = f(x) - b
                # written using the old error so we don't recompute the whole sum
                b1 = (
                    self.bias - err_i
                    - y[i] * (self.alphas[i] - old_i) * self.K[i, i]
                    - y[j] * (self.alphas[j] - old_j) * self.K[i, j]
                )
                b2 = (
                    self.bias - err_j
                    - y[i] * (self.alphas[i] - old_i) * self.K[i, j]
                    - y[j] * (self.alphas[j] - old_j) * 1
                )

                # if an alpha is strictly between 0 and C,
                # its point is a free support vector lying on the margin
                if 0 < self.alphas[i] < self.C:
                    self.bias = b1

                elif 0 < self.alphas[j] < self.C:
                    self.bias = b2

                else:
                    self.bias = (b1 + b2) / 2.0

                num_changed_alphas += 1

            # this has to happen AFTER checking all training points
            if num_changed_alphas == 0:
                passes += 1
            else:
                passes = 0

        # only points with non-zero alpha contribute during prediction
        support_mask = self.alphas > 1e-5

        self.support_vectors = self.x_tr[
            support_mask
        ]

        self.support_alphas = self.alphas[
            support_mask
        ]

        self.support_y = self.y_tr[
            support_mask
        ]

        return self

    def predict(self, x):
        predictions = []

        for point in x:
            score = 0.0

            # f(x) = sum(alpha_i * y_i * K(x_i, x)) + bias
            # only support vectors need to be considered
            for alpha, y_i, support_vector in zip(
                self.support_alphas,
                self.support_y,
                self.support_vectors
            ):
                score += (
                    alpha
                    * y_i
                    * self._rbf_func(support_vector, point)
                )

            score += self.bias

            predictions.append(
                1.0 if score >= 0 else -1.0
            )

        return np.array(predictions)


load_data()
model = Kernel_SVM()

def train_test_split(x, y, test_size=0.2):
    n = x.shape[0]

    indices = np.arange(n)
    np.random.shuffle(indices)

    split = int((1 - test_size) * n)

    train_idx = indices[:split]
    test_idx = indices[split:]

    return (
        x[train_idx],
        x[test_idx],
        y[train_idx],
        y[test_idx]
    )

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

x_train, x_test, y_train, y_test = train_test_split(x, y)
mean = np.mean(x_train, axis=0)
std = np.std(x_train, axis=0)

x_train = (x_train - mean) / std
x_test = (x_test - mean) / std
model.fit(x_train, y_train)

train_pred = model.predict(x_train)
test_pred = model.predict(x_test)

print(f"Train Accuracy: {accuracy(y_train, train_pred):.4f}")
print(f"Test Accuracy : {accuracy(y_test, test_pred):.4f}")

Train Accuracy: 0.9890
Test Accuracy : 0.9474
